# Step 2. L2 Logistic Regression, kNN, and XGBoost with TF-IDF and E5

Train three score classifiers: L2-regularized logistic classification, kNN,
and XGBoost. Compare **TF-IDF** and frozen **E5 with four extra features
(388 dimensions)**. Each question has a score distribution for each of the
11 candidate LLMs over the fixed values **0, 0.25, 0.5, 0.75 and 1**.

Read each distribution in two ways:

- **Hard class:** the score with the highest predicted probability.
- **Expected score:** the probability-weighted average of the five scores.

Both outputs share the same predicted dollar cost. The experiment uses train,
calibration and validation. Training-only cross-validation selects the logistic
regularization strength and the Ridge cost alpha. A separate kNN CV selects
score and cost neighbor counts using the same grouped training folds.
Calibration is the development split for comparing these predictors; no
probability calibrator is fitted here. Validation predictions are saved for
downstream routing-policy development. The test split is reserved for the
final complete system.

For a complete run, restart the kernel, keep `RUN_MODE="full"`, and choose
**Run All**. All predictors and their training-CV searches are recomputed from
the prepared inputs. Full results replace the notebook's fixed `full/` output
directory. Smoke checks use separate timestamped directories.

Use `experiment_manifest.json` for the selected settings and `run_status.json`
to confirm that the output export completed.


## 1. Environment and paths

Use a Python environment with NumPy, pandas, SciPy, scikit-learn >= 1.6,
XGBoost >= 2.0, joblib, IPython and Jinja2. The prepared preprocessing objects
should be used with compatible package versions.

Keep the prepared files in this layout:

```text
project/
  data/processed/
    char_tfidf_50k/
    multilingual_e5_small_chunk480/
  2_Router_Training/
    TF-IDF_E5_Score_Classification_Router.ipynb
```

A layout with `1_data_preparation/data/processed/` is also supported. The code
searches the current working directory and its parents. If the data lives
elsewhere, set `DATA_PREPARATION_DIR` to the folder containing `data/`, or set
the `ERDOS_DATA_DIR` environment variable. No user-specific paths are required.
Copy the prepared data with the notebook, or run the project's data-preparation
steps before using this file.

`RUN_MODE="full"` uses all prepared rows. Restart the kernel and run every
cell in order. The configuration cell clears and recreates only the owned
`experiment_outputs/score_classification_5class/full/` directory before fitting.
The next full run replaces its previous results, including files no longer
produced by the current settings. Timestamped historical runs stay untouched.
The first run creates `full/`; an unrecognized existing folder is never cleared.

`"smoke"` runs a small execution check in a separate timestamped folder and
does not touch `full/`. Its metrics are not a performance estimate. If a run
stops early, `run_status.json` remains `"running"`; only a finished export marks
it `"completed"`. A new full run can reset an owned incomplete directory.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import inspect
import json
import os
import shutil
import stat
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier, XGBRegressor


In [2]:
RUN_MODE = "full"
DATA_PREPARATION_DIR = None  # Optional: folder containing data/.
OUTPUT_BASE_OVERRIDE = None  # Optional: output folder outside the project.
SEED = 42
SPLITS = ("train", "calibration", "validation")
# Base-predictor diagnostics use calibration; validation feeds routing policies.
EVALUATION_SPLIT = "calibration"
SMOKE_LIMITS = {"train": 512, "calibration": 128, "validation": 256}

score_support = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
K_VALUES = [1, 2, 3, 5, 10, 20, 40, 60, 80, 100]
QUERY_BATCH_SIZE = 128

# Independent LLM fits run in separate processes; use 1 for serial execution.
FIT_N_JOBS = 2

LINEAR_C_GRID = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])
LINEAR_MAX_ITER = 300
LINEAR_RETRY_MAX_ITER = (1000, 2000)
LINEAR_TOL = 1e-3

XGB_DEVICE = "cpu"  # Set to "cuda" with a supported GPU installation.
XGB_N_JOBS = 4
XGB_ROUNDS = 10 if RUN_MODE == "smoke" else 100

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if XGB_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'.")


def resolve_preparation_dir(override=None):
    """Find a prepared-data folder, preferring the closest project root."""
    requested = override or os.environ.get("ERDOS_DATA_DIR")
    if requested:
        candidates = [Path(requested).expanduser().resolve()]
    else:
        working_dir = Path.cwd().resolve()
        candidates = [
            candidate
            for parent in (working_dir, *working_dir.parents)
            for candidate in (parent, parent / "1_data_preparation")
        ]
    required = (
        Path("data/processed/char_tfidf_50k/preprocessing.joblib"),
        Path("data/processed/multilingual_e5_small_chunk480/embedding_metadata.json"),
    )
    for candidate in dict.fromkeys(candidates):
        if all((candidate / relative).is_file() for relative in required):
            return candidate.resolve()
    raise FileNotFoundError(
        "Prepared TF-IDF and E5 files were not found. Set DATA_PREPARATION_DIR "
        "or ERDOS_DATA_DIR to the folder containing data/processed/."
    )


def prepare_output_directory(base, mode, stamp):
    """Reset only an owned full-output directory; keep smoke runs separate."""
    base = Path(base).expanduser().resolve()
    if base == Path(base.anchor):
        raise ValueError("The output base must be a project folder, not a filesystem root.")
    if mode not in {"full", "smoke"}:
        raise ValueError("Output mode must be 'full' or 'smoke'.")
    target = base / ("full" if mode == "full" else f"smoke_seed{SEED}_{stamp}")

    # Inspect the original path before resolving it, so links cannot hide the target.
    try:
        target_info = target.lstat()
    except FileNotFoundError:
        target_info = None
    reparse_flag = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0)
    if target_info is not None and (
        target.is_symlink()
        or getattr(target_info, "st_file_attributes", 0) & reparse_flag
    ):
        raise ValueError(f"Output directory cannot be a symlink or junction: {target}")
    if target.resolve() != base / target.name or target.parent != base:
        raise ValueError(f"Output directory must stay directly inside the output base: {target}")
    if target_info is not None and not stat.S_ISDIR(target_info.st_mode):
        raise ValueError(f"Output path is not a directory: {target}")

    workflow = "erdos_score_classification_router"
    if mode == "full" and target_info is not None:
        # The marker prevents deleting an unrelated folder named 'full'.
        marker = target / "run_status.json"
        try:
            previous = json.loads(marker.read_text(encoding="utf-8"))
        except (OSError, ValueError) as error:
            raise ValueError(f"Refusing to replace an unrecognized output folder: {target}") from error
        if previous.get("workflow") != workflow or previous.get("mode") != "full":
            raise ValueError(f"Refusing to replace an unrelated output folder: {target}")
        print("Replacing previous full-training results:", target)
        # target is an absolute, checked direct child of base; no other run is removed.
        shutil.rmtree(target)

    target.mkdir(parents=True, exist_ok=False)
    status = {
        "workflow": workflow, "mode": mode, "status": "running",
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "output_policy": "replace_fixed_full" if mode == "full" else "new_smoke_directory",
    }
    (target / "run_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    return target


preparation_dir = resolve_preparation_dir(DATA_PREPARATION_DIR)
table_dir = preparation_dir / "data/processed/char_tfidf_50k"
e5_dir = preparation_dir / "data/processed/multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (
    Path(OUTPUT_BASE_OVERRIDE).expanduser().resolve()
    if OUTPUT_BASE_OVERRIDE is not None
    else project_dir / "2_Router_Training/experiment_outputs/score_classification_5class"
)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = prepare_output_directory(output_base, RUN_MODE, run_stamp)
# Reset provenance on every run, including when an existing kernel is reused.
execution_provenance = {
    "refit_scope": "all_predictors",
    "prepared_inputs": "Step 1 TF-IDF and frozen E5 features",
    "output_policy": "replace_fixed_full" if RUN_MODE == "full" else "new_smoke_directory",
}
print("Mode:", RUN_MODE)
print("Prepared data:", preparation_dir)
print("Outputs:", run_dir)


Mode: full
Prepared data: C:\Users\johnh\MachineLearning\project_erdos
Outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full


## 2. Load the prepared inputs

Reuse the question, score and dollar-cost tables, pure TF-IDF features, and
388-dimensional E5 features. Row indices and LLM column order must agree.
The original score tables stay unchanged throughout this notebook.


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
tfidf_vectorizer_50k = preprocessing["tfidf_vectorizer"]
model_names = list(preprocessing["model_names"])
assert len(model_names) == 11
n_tfidf_features = len(tfidf_vectorizer_50k.vocabulary_)
e5_length_scaler = joblib.load(e5_dir / "length_scaler.joblib")
e5_count_scaler = joblib.load(e5_dir / "count_scaler.joblib")
feature_labels = {"tfidf": "TF-IDF", "e5_388": "E5 + features (388)"}
full_tables, full_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    full_tables[split] = {name: pd.read_pickle(table_dir / f"{split}_{name}.pkl")
                         for name in ("questions", "scores", "costs")}
    q, s, c = (full_tables[split][name] for name in ("questions", "scores", "costs"))
    assert q.index.is_unique and q.index.equals(s.index) and q.index.equals(c.index)
    assert s.columns.tolist() == c.columns.tolist() == model_names
    assert np.isfinite(s.to_numpy(dtype=float)).all() and s.ge(0).all().all() and s.le(1).all().all()
    assert np.isfinite(c.to_numpy(dtype=float)).all() and c.gt(0).all().all()
    full_features["tfidf"][split] = load_npz(table_dir / f"X_{split}.npz")[:, :n_tfidf_features].astype(np.float32).tocsr()
    full_features["e5_388"][split] = np.load(e5_dir / f"X_{split}_e5_full_features.npy", allow_pickle=False)
    assert full_features["tfidf"][split].shape == (len(q), n_tfidf_features)
    assert full_features["e5_388"][split].shape == (len(q), 388)

# Keep the full tables for label coverage; only the execution subset changes.
tables, split_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    q = full_tables[split]["questions"]
    positions = np.arange(len(q))
    if RUN_MODE == "smoke":
        rng = np.random.default_rng(SEED + SPLITS.index(split))
        positions = rng.choice(len(q), min(SMOKE_LIMITS[split], len(q)), replace=False)
        if split == "train":
            # Include each observed score to exercise the fixed mapping in smoke mode.
            scores = full_tables[split]["scores"].to_numpy(dtype=float)
            examples = [np.argwhere(scores == value)[0, 0] for value in np.unique(scores)]
            positions = np.unique(np.r_[positions, examples])
        positions.sort()
    tables[split] = {name: frame.iloc[positions].copy() for name, frame in full_tables[split].items()}
    for feature in feature_labels:
        split_features[feature][split] = full_features[feature][split][positions].copy()
    tables[split]["questions"][["sample_id"]].to_csv(run_dir / f"{split}_rows.csv")
train_questions, train_scores, train_costs = (tables["train"][name] for name in ("questions", "scores", "costs"))
evaluation_scores, evaluation_costs = (tables[EVALUATION_SPLIT][name] for name in ("scores", "costs"))
train_scores_arr = train_scores.to_numpy(dtype=float)  # Original recorded scores.
cost_means = train_costs.to_numpy(dtype=float).mean(axis=0)
train_cost_normalized = (train_costs.to_numpy(dtype=float) / cost_means).astype(np.float32)
train_groups = train_questions["question_group"].to_numpy()
assert len(train_questions) >= max(K_VALUES)
all_questions = pd.concat([tables[split]["questions"].assign(_split=split) for split in SPLITS])
assert all_questions["sample_id"].is_unique
for group_key in ("question_group", "question_text"):
    assert all_questions.groupby(group_key)["_split"].nunique().eq(1).all()
display(pd.DataFrame({"Questions": {split: len(tables[split]["questions"]) for split in SPLITS}}))
print("Fixed score classes:", score_support.tolist())


,Questions
train,21900
calibration,3648
validation,3649


Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]


## 3. Why five score classes?

The five anchors cover **99.858% of training scores**. The table below counts
individual question-LLM scores in the prepared files, before any mapping or
smoke subsetting. The other values are sparse and provide little support for
learning separate classes.

| Recorded score | Train count | Calibration count |
|---|---:|---:|
| 0 | 93,268 | 15,532 |
| 0.25 | 15,064 | 2,417 |
| 0.5 | 8,075 | 1,304 |
| 0.75 | 27,679 | 4,599 |
| 1 | 96,473 | 16,203 |
| Other values | 341 | 73 |
| **Five-anchor coverage** | **99.858%** | **99.818%** |

Using five classes gives a compact common output schema and reduces the number
of sparsely supported categories. It is a practical simplification; it does not
guarantee higher predictive accuracy or resolve uncertainty in the grading.
The next cell recomputes this table from the files loaded on your computer.

For classification, map each recorded score $s$ to its nearest fixed anchor:

$$z(s)=\arg\min_{j\in\{0,\ldots,4\}}|s-a_j|,
\qquad a=(0,\,0.25,\,0.5,\,0.75,\,1).$$

An exact midpoint uses the lower anchor. Current non-anchor values map as follows:

| Recorded values | Classification target |
|---|---:|
| 0.1 | 0 |
| 0.2, 0.3 | 0.25 |
| 0.4, 0.6 | 0.5 |
| 0.7, 0.8 | 0.75 |
| 0.9 | 1 |

The anchors and rule are fixed before evaluation. All questions remain in the
experiment. Classification accuracy uses mapped targets. Score MSE and MAE use
the **original recorded scores**, so their meaning is preserved.


In [4]:
coverage_rows = []
for label in [*score_support, "Other values"]:
    row = {"Recorded score": label}
    for split in ("train", EVALUATION_SPLIT):
        values = full_tables[split]["scores"].to_numpy(dtype=float)
        row[split] = int(
            (~np.isin(values, score_support)).sum()
            if isinstance(label, str) else (values == label).sum()
        )
    coverage_rows.append(row)
label_coverage_counts = pd.DataFrame(coverage_rows).set_index("Recorded score")
anchor_coverage = pd.DataFrame({
    split: {
        "Total score entries": full_tables[split]["scores"].size,
        "Five-anchor coverage": np.isin(
            full_tables[split]["scores"].to_numpy(dtype=float), score_support,
        ).mean(),
    }
    for split in ("train", EVALUATION_SPLIT)
}).T
display(label_coverage_counts)
display(anchor_coverage.style.format({
    "Total score entries": "{:,.0f}", "Five-anchor coverage": "{:.3%}",
}))
label_coverage_counts.to_csv(run_dir / "raw_score_value_counts.csv")
anchor_coverage.to_csv(run_dir / "raw_score_anchor_coverage.csv")


,train,calibration
Recorded score,,
0.0,93268,15532
0.25,15064,2417
0.5,8075,1304
0.75,27679,4599
1.0,96473,16203
Other values,341,73


,Total score entries,Five-anchor coverage
train,"240,900",99.858%
calibration,"40,128",99.818%


In [5]:
def encode_score_classes(values):
    """Map original scores to fixed anchors without changing the source array."""
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Scores must be finite and lie in [0, 1].")
    distances = np.abs(values[..., None] - score_support)
    return distances.argmin(axis=-1).astype(np.int64)


def align_class_probabilities(probabilities, classes, n_classes):
    probabilities = np.asarray(probabilities, dtype=float)
    classes = np.asarray(classes, dtype=int)
    aligned = np.zeros((len(probabilities), n_classes), dtype=float)
    aligned[:, classes] = probabilities
    if not np.isfinite(aligned).all() or np.any(aligned < 0) or not np.allclose(aligned.sum(axis=1), 1, atol=1e-6):
        raise ValueError("Invalid score probability distribution.")
    return aligned


def class_log_loss(targets, probabilities, floor=1e-12):
    true_probability = probabilities[np.arange(len(targets)), targets]
    return float(-np.log(np.maximum(true_probability, floor)).mean())


def prediction_frame(values, split):
    return pd.DataFrame(values, index=tables[split]["questions"].index, columns=model_names)


score_class_ids = encode_score_classes(train_scores_arr)
training_folds = list(GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    train_scores_arr, groups=train_groups,
))
for fit_indices, heldout_indices in training_folds:
    assert not np.intersect1d(train_groups[fit_indices], train_groups[heldout_indices]).size
e5_train = split_features["e5_388"]["train"]
raw_lengths = e5_length_scaler.inverse_transform(e5_train[:, 384:385])
raw_counts = e5_count_scaler.inverse_transform(e5_train[:, 385:387])
training_texts = train_questions["question_text"].to_numpy()


def build_fold_features(feature, fit_indices, heldout_indices):
    if feature == "tfidf":
        vectorizer = clone(tfidf_vectorizer_50k)
        return (
            vectorizer.fit_transform(training_texts[fit_indices]).astype(np.float32).tocsr(),
            vectorizer.transform(training_texts[heldout_indices]).astype(np.float32).tocsr(),
        )
    length_scaler = clone(e5_length_scaler).fit(raw_lengths[fit_indices])
    count_scaler = clone(e5_count_scaler).fit(raw_counts[fit_indices])

    def transform(positions):
        return np.column_stack([e5_train[positions, :384],
                                length_scaler.transform(raw_lengths[positions]),
                                count_scaler.transform(raw_counts[positions]),
                                e5_train[positions, 387:388]]).astype(np.float32)

    return transform(fit_indices), transform(heldout_indices)


prediction_bundles = {}


## 4. L2-regularized logistic classification

Train one classifier per candidate LLM. For one LLM, let $N$ be the number of
training questions, $z_i$ the mapped score-class index of question $i$, and
$p_{i,z_i}$ its predicted probability for that class. Let $W$ contain the
class-weight vectors as rows; $\|W\|_F^2$ is the sum of their squared entries.
With regularization coefficient $\lambda$, the training objective is

$$\mathcal L_{\mathrm{score}}=
-\frac1N\sum_{i=1}^N\log p_{i,z_i}
+\frac\lambda2\|W\|_F^2.$$

Intercepts are unpenalized. Scikit-learn uses $C>0$, with
$\lambda=1/(NC)$ in this mean-loss formulation. Smaller C gives stronger
regularization.

Select one `C` per
feature set by mean cross-entropy over three training folds and the 11 LLMs.
Test C = 0.01, 0.1, 1, 10, 100, 1000 and 10000. This is a broad,
logarithmically spaced search. NLL selects C; Accuracy is an auxiliary metric.
Question groups stay within one fold. TF-IDF and the extra-feature scalers are
refitted inside each fold; the frozen E5 vectors are reused.

The price head is a multi-output Ridge model. For LLM $m$, let
$\mu_m=\operatorname{mean}_{\mathrm{train}}c_{im}$. Fit normalized costs
$c_{im}/\mu_m$ with a squared-error objective and L2 penalty. Clip negative
estimates to a small positive value and multiply by $\mu_m$ to restore dollars.

The score-fit cell trains the selected Logistic classifiers. The Ridge
cost-fit cell then trains the CV-selected cost heads and assembles their shared
probability/cost bundles. A full run fits both parts from the prepared data.
Final Logistic classifiers must converge before evaluation and export continue.


In [6]:
L2_OPTIONS = {"penalty": "l2"} if inspect.signature(LogisticRegression).parameters["l1_ratio"].default is None else {"l1_ratio": 0.0}


def fit_linear_score_head(features, targets, C):
    """Return a classifier, or its constant class, with fit diagnostics."""
    classes = np.unique(targets)
    if classes.size == 1:
        return {
            "classifier": None, "constant_class": int(classes[0]),
            "iterations": 0, "converged": True, "observed_classes": classes,
            "convergence_warning_count": 0,
        }
    classifier = LogisticRegression(
        C=float(C), solver="lbfgs", max_iter=LINEAR_MAX_ITER, **L2_OPTIONS,
        tol=LINEAR_TOL, random_state=SEED,
    )
    total_iterations, warning_count = 0, 0
    for iteration_limit in (LINEAR_MAX_ITER, *LINEAR_RETRY_MAX_ITER):
        classifier.set_params(max_iter=iteration_limit)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(features, targets)
        convergence_messages = [str(item.message) for item in caught
                                if issubclass(item.category, ConvergenceWarning)]
        warning_count += len(convergence_messages)
        total_iterations += int(classifier.n_iter_.max())
        for item in caught:
            if not issubclass(item.category, ConvergenceWarning):
                warnings.warn(str(item.message), item.category)
        if not convergence_messages:
            break
        # Continue from the current coefficients with the same objective and tolerance.
        classifier.set_params(warm_start=True)
        print(f"    iteration limit {iteration_limit}: continuing optimization", flush=True)
    if not np.isfinite(classifier.coef_).all() or not np.isfinite(classifier.intercept_).all():
        raise ValueError("A linear score classifier produced nonfinite parameters.")
    return {
        "classifier": classifier, "constant_class": None,
        "iterations": total_iterations,
        "converged": not convergence_messages,
        "observed_classes": classifier.classes_.astype(int),
        "convergence_warning_count": warning_count,
    }


def linear_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["constant_class"]] = 1.0
        return probabilities
    classifier = head["classifier"]
    return align_class_probabilities(
        classifier.predict_proba(features), classifier.classes_,
        len(score_support),
    )


def fit_linear_score_job(features, targets, C):
    """Fit one independent LLM score head and record its own training time."""
    started = perf_counter()
    head = fit_linear_score_head(features, targets, C)
    return head, perf_counter() - started


In [7]:
linear_cv_rows = []
linear_cv_head_rows = []
for feature in feature_labels:
    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        print(f"Linear score CV / {feature_labels[feature]} / fold {fold_id}: features", flush=True)
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        for C in LINEAR_C_GRID:
            started = perf_counter()
            losses, accuracies = [], []
            unseen_labels = 0
            n_unconverged = 0
            warning_count = 0
            max_iterations = 0
            max_observed_classes = 0
            fitted_heads = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                           max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
                joblib.delayed(fit_linear_score_job)(
                    X_fit, score_class_ids[fit_indices, position], C,
                ) for position in range(len(model_names))
            )
            for model_position, model_name in enumerate(model_names):
                head, fit_seconds = fitted_heads[model_position]
                probabilities = linear_score_probabilities(head, X_heldout)
                targets = score_class_ids[heldout_indices, model_position]
                losses.append(class_log_loss(targets, probabilities))
                accuracies.append(float((probabilities.argmax(axis=1) == targets).mean()))
                linear_cv_head_rows.append({
                    "Feature": feature, "Fold": fold_id, "C": float(C),
                    "Model": model_name, "Questions": len(heldout_indices),
                    "NLL": losses[-1], "Accuracy": accuracies[-1],
                    "Iterations": head["iterations"], "Converged": head["converged"],
                    "Convergence_warnings": head["convergence_warning_count"],
                    "Seconds": fit_seconds,
                })
                unseen_labels += int((~np.isin(targets, head["observed_classes"])).sum())
                n_unconverged += int(not head["converged"])
                warning_count += head["convergence_warning_count"]
                max_iterations = max(max_iterations, head["iterations"])
                max_observed_classes = max(max_observed_classes, len(head["observed_classes"]))
                print(
                    f"  C={C:g} / {model_position + 1}/{len(model_names)} / "
                    f"{model_name}: CE={losses[-1]:.4f}, "
                    f"iterations={head['iterations']}, converged={head['converged']}",
                    flush=True,
                )
            linear_cv_rows.append({
                "Feature": feature, "Fold": fold_id, "C": float(C),
                "Questions": len(heldout_indices), "Mean_CrossEntropy": np.mean(losses),
                "Mean_Accuracy": np.mean(accuracies),
                "Unseen_label_events": unseen_labels,
                "Unconverged_heads": n_unconverged, "Max_iterations": max_iterations,
                "Convergence_warnings": warning_count, "Max_observed_classes": max_observed_classes,
                "Seconds": perf_counter() - started,
            })
            # Persist completed candidate/fold diagnostics during the long search.
            pd.DataFrame(linear_cv_rows).to_csv(run_dir / "linear_cv_folds.csv", index=False)
            pd.DataFrame(linear_cv_head_rows).to_csv(run_dir / "linear_cv_heads.csv", index=False)
        del X_fit, X_heldout

linear_cv_table = pd.DataFrame(linear_cv_rows)
linear_cv_summary_rows = []
for (feature, C), rows in linear_cv_table.groupby(["Feature", "C"], sort=False):
    linear_cv_summary_rows.append({
        "Feature": feature, "C": C,
        "CV_CrossEntropy": np.average(rows["Mean_CrossEntropy"], weights=rows["Questions"]),
        "CV_Accuracy": np.average(rows["Mean_Accuracy"], weights=rows["Questions"]),
        "Unseen_label_events": int(rows["Unseen_label_events"].sum()),
        "Unconverged_heads": int(rows["Unconverged_heads"].sum()),
        "Convergence_warnings": int(rows["Convergence_warnings"].sum()),
        "Max_iterations": int(rows["Max_iterations"].max()),
        "Max_observed_classes": int(rows["Max_observed_classes"].max()),
        "Seconds": rows["Seconds"].sum(),
    })
linear_cv_summary = pd.DataFrame(linear_cv_summary_rows)
linear_cv_head_table = pd.DataFrame(linear_cv_head_rows)
linear_cv_head_table.to_csv(run_dir / "linear_cv_heads.csv", index=False)
linear_cv_summary.to_csv(run_dir / "linear_cv_summary.csv", index=False)
if linear_cv_summary["Unconverged_heads"].sum():
    raise RuntimeError("Resolve unconverged CV fits before selecting C; see linear_cv_heads.csv.")
selected_linear_C = {}
for feature in feature_labels:
    choices = linear_cv_summary.loc[linear_cv_summary["Feature"] == feature]
    # Exact ties favor the smaller C, which gives stronger regularization.
    best = choices.sort_values(["CV_CrossEntropy", "C"], kind="stable").iloc[0]
    selected_linear_C[feature] = float(best["C"])
display(linear_cv_summary)
print("Selected score-classifier C:", selected_linear_C)


Linear score CV / TF-IDF / fold 1: features
  C=0.01 / 1/11 / WizardLM/WizardLM-13B-V1.2: CE=0.8760, iterations=12, converged=True
  C=0.01 / 2/11 / claude-instant-v1: CE=0.8181, iterations=13, converged=True
  C=0.01 / 3/11 / claude-v1: CE=0.7928, iterations=15, converged=True
  C=0.01 / 4/11 / claude-v2: CE=0.8040, iterations=15, converged=True
  C=0.01 / 5/11 / gpt-3.5-turbo-1106: CE=0.8592, iterations=17, converged=True
  C=0.01 / 6/11 / gpt-4-1106-preview: CE=0.6565, iterations=12, converged=True
  C=0.01 / 7/11 / meta/code-llama-instruct-34b-chat: CE=0.6435, iterations=10, converged=True
  C=0.01 / 8/11 / meta/llama-2-70b-chat: CE=0.7570, iterations=14, converged=True
  C=0.01 / 9/11 / mistralai/mistral-7b-chat: CE=0.7927, iterations=14, converged=True
  C=0.01 / 10/11 / mistralai/mixtral-8x7b-chat: CE=0.8824, iterations=16, converged=True
  C=0.01 / 11/11 / zero-one-ai/Yi-34B-Chat: CE=0.8220, iterations=12, converged=True
  C=0.1 / 1/11 / WizardLM/WizardLM-13B-V1.2: CE=0.7663, i

,Feature,C,CV_CrossEntropy,CV_Accuracy,Unseen_label_events,Unconverged_heads,Convergence_warnings,Max_iterations,Max_observed_classes,Seconds
0,tfidf,0.01,0.790557,0.662989,0,0,0,17,5,57.481738
1,tfidf,0.10,0.672715,0.679232,0,0,0,31,5,80.433828
2,tfidf,1.00,0.644155,0.681826,0,0,0,71,5,136.100904
3,tfidf,10.00,0.740185,0.658016,0,0,0,120,5,240.458609
4,tfidf,100.00,1.059246,0.637742,0,0,0,139,5,349.108576
5,tfidf,1000.00,1.460150,0.634736,0,0,0,143,5,403.165447
6,tfidf,10000.00,1.535908,0.632603,0,0,0,138,5,357.643520
7,e5_388,0.01,0.705808,0.659128,0,0,0,28,5,7.538864
8,e5_388,0.10,0.666148,0.676359,0,0,0,47,5,11.684394
9,e5_388,1.00,0.641868,0.684027,0,0,0,91,5,20.308669


Selected score-classifier C: {'tfidf': 1.0, 'e5_388': 1.0}


### Cross-validation for Ridge alpha

Select one Ridge alpha for each representation using the same three grouped
training folds. Candidate values are **0.01, 0.1, 1, 10, 100, 1000 and 10000**.

The selection metric is **mean Cost WAPE across the 11 LLMs**. Let $N$ be the
number of training questions, $m=1,\ldots,11$ index LLMs, and $c_{im}$ be the
recorded dollar cost for training question $i$. For candidate penalty $\alpha$,
let $\hat c_{im}^{(\alpha)}$ be the dollar prediction made while question $i$ is
held out. Every training question receives one such out-of-fold prediction.
Define the cost-selection criterion as

$$
\mathrm{CV\_CostWAPE}(\alpha)=\frac1{11}\sum_{m=1}^{11}
\frac{\sum_{i=1}^{N}|\hat c_{im}^{(\alpha)}-c_{im}|}
{\sum_{i=1}^{N}c_{im}}.
$$

Lower is better. Aggregate dollar errors over all held-out questions before
forming each LLM's ratio. The individual fold WAPEs are diagnostics. Exact ties
favor larger alpha. A best value at a grid boundary calls for a wider search.

Within each fold, refit TF-IDF or the E5 auxiliary-feature scalers on the fitting
rows. Normalize each LLM's fitting costs by its **fold-fitting mean**, predict
normalized costs, apply the positive floor, and restore dollars using that same
mean. Calibration, validation and test labels do not participate in selection.

Save the CV tables, row IDs, out-of-fold predictions and selection settings.
The Ridge cost-fit cell then trains the final heads using `selected_cost_ridge_alpha`.


In [8]:
# Select Ridge alpha with grouped cross-validation on training data.
COST_RIDGE_ALPHA_GRID = np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])
cost_cv_targets = train_costs.loc[:, model_names].to_numpy(dtype=np.float64)
cost_cv_n_questions, cost_cv_n_models = cost_cv_targets.shape
assert cost_cv_n_models == 11
assert train_costs.index.equals(train_questions.index)
assert np.isfinite(cost_cv_targets).all() and (cost_cv_targets > 0).all()
assert len(training_folds) == 3

# Every training question must be held out once, with no group leakage.
cost_cv_holdout_counts = np.zeros(cost_cv_n_questions, dtype=int)
cost_cv_fold_ids = np.zeros(cost_cv_n_questions, dtype=int)
for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
    assert not np.intersect1d(fit_indices, heldout_indices).size
    assert len(fit_indices) + len(heldout_indices) == cost_cv_n_questions
    assert not np.intersect1d(train_groups[fit_indices], train_groups[heldout_indices]).size
    cost_cv_holdout_counts[heldout_indices] += 1
    cost_cv_fold_ids[heldout_indices] = fold_id
assert np.all(cost_cv_holdout_counts == 1)

if RUN_MODE == "smoke":
    warnings.warn("Cost CV is using smoke rows. Use full mode before interpreting alpha choices.")

cost_cv_output_dir = run_dir / "cost_ridge_cv"
cost_cv_output_dir.mkdir(parents=True, exist_ok=True)
cost_cv_fold_rows, cost_cv_model_rows, cost_cv_summary_rows = [], [], []
cost_cv_oof_predictions = {}
cost_cv_started = perf_counter()

for feature in feature_labels:
    # Store dollar predictions in original training-row order for each alpha.
    predictions = np.full(
        (len(COST_RIDGE_ALPHA_GRID), cost_cv_n_questions, cost_cv_n_models),
        np.nan, dtype=np.float64,
    )
    normalized_sse = np.zeros(len(COST_RIDGE_ALPHA_GRID), dtype=np.float64)
    fit_seconds = np.zeros(len(COST_RIDGE_ALPHA_GRID), dtype=np.float64)

    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        print(f"Ridge CV / {feature_labels[feature]} / fold {fold_id}: prepare features", flush=True)
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        fold_cost_means = cost_cv_targets[fit_indices].mean(axis=0)
        fit_targets = (cost_cv_targets[fit_indices] / fold_cost_means).astype(np.float32)
        heldout_costs = cost_cv_targets[heldout_indices]
        assert X_fit.shape[0] == len(fit_indices)
        assert X_heldout.shape[0] == len(heldout_indices)

        for alpha_position, alpha in enumerate(COST_RIDGE_ALPHA_GRID):
            started = perf_counter()
            cost_regressor = Ridge(alpha=float(alpha), solver="lsqr", tol=1e-3, max_iter=1000)
            cost_regressor.fit(X_fit, fit_targets)
            predicted_costs = np.maximum(cost_regressor.predict(X_heldout), 1e-6) * fold_cost_means
            assert predicted_costs.shape == heldout_costs.shape
            assert np.isfinite(predicted_costs).all() and (predicted_costs > 0).all()
            predictions[alpha_position, heldout_indices] = predicted_costs

            errors = predicted_costs - heldout_costs
            normalized_sse[alpha_position] += np.square(errors / fold_cost_means).sum()
            elapsed = perf_counter() - started
            fit_seconds[alpha_position] += elapsed
            fold_wape = np.mean(np.abs(errors).sum(axis=0) / heldout_costs.sum(axis=0))
            iterations = int(np.max(cost_regressor.n_iter_))
            cost_cv_fold_rows.append({
                "Feature": feature, "Fold": fold_id, "alpha": float(alpha),
                "Fit_questions": len(fit_indices), "Heldout_questions": len(heldout_indices),
                "Fold_Cost_WAPE": float(fold_wape),
                "Fold_Cost_MSE_normalized": float(np.square(errors / fold_cost_means).mean()),
                "Max_iterations": iterations, "Seconds": elapsed,
            })
            # Fold WAPEs are diagnostic; selection uses all OOF rows below.
            print(f"  alpha={alpha:g}: fold WAPE={fold_wape:.3%}, iterations={iterations}, seconds={elapsed:.1f}", flush=True)
            pd.DataFrame(cost_cv_fold_rows).to_csv(cost_cv_output_dir / "cost_cv_folds.csv", index=False)
            if iterations >= 1000:
                warnings.warn(f"Cost Ridge reached the iteration limit: {feature}, fold {fold_id}, alpha {alpha:g}.")

        del X_fit, X_heldout, cost_regressor

    assert np.isfinite(predictions).all()
    cost_cv_oof_predictions[feature] = predictions
    for alpha_position, alpha in enumerate(COST_RIDGE_ALPHA_GRID):
        errors = predictions[alpha_position] - cost_cv_targets
        # Sum dollar errors across every held-out row BEFORE forming WAPE ratios.
        model_wapes = np.abs(errors).sum(axis=0) / cost_cv_targets.sum(axis=0)
        cost_cv_summary_rows.append({
            "Feature": feature, "alpha": float(alpha),
            "CV_Cost_WAPE": float(model_wapes.mean()),
            "CV_Cost_MAE": float(np.abs(errors).mean()),
            "CV_Cost_MSE": float(np.square(errors).mean()),
            "CV_Cost_MSE_normalized": float(normalized_sse[alpha_position] / cost_cv_targets.size),
            "Fit_seconds": float(fit_seconds[alpha_position]),
        })
        for model_position, model_name in enumerate(model_names):
            cost_cv_model_rows.append({
                "Feature": feature, "alpha": float(alpha), "Model": model_name,
                "CV_Cost_WAPE": float(model_wapes[model_position]),
                "CV_Cost_MAE": float(np.abs(errors[:, model_position]).mean()),
                "CV_Cost_MSE": float(np.square(errors[:, model_position]).mean()),
            })

cost_cv_summary = pd.DataFrame(cost_cv_summary_rows)
selected_cost_ridge_alpha = {}
for feature in feature_labels:
    choices = cost_cv_summary.loc[cost_cv_summary["Feature"].eq(feature)]
    # Exact ties favor larger alpha.
    best = choices.sort_values(["CV_Cost_WAPE", "alpha"], ascending=[True, False], kind="stable").iloc[0]
    selected_cost_ridge_alpha[feature] = float(best["alpha"])
    if best["alpha"] in (COST_RIDGE_ALPHA_GRID.min(), COST_RIDGE_ALPHA_GRID.max()):
        warnings.warn(f"Best cost alpha is on the grid boundary for {feature}; consider extending the grid.")

cost_cv_summary["Selected"] = [
    alpha == selected_cost_ridge_alpha[feature]
    for feature, alpha in cost_cv_summary[["Feature", "alpha"]].itertuples(index=False, name=None)
]
cost_cv_summary.to_csv(cost_cv_output_dir / "cost_cv_summary.csv", index=False)
pd.DataFrame(cost_cv_model_rows).to_csv(cost_cv_output_dir / "cost_cv_per_llm.csv", index=False)
cost_cv_rows = train_questions.loc[:, ["sample_id", "question_group"]].copy()
cost_cv_rows["Fold"] = cost_cv_fold_ids
cost_cv_rows.to_csv(cost_cv_output_dir / "cost_cv_training_rows.csv")
for feature, predictions in cost_cv_oof_predictions.items():
    np.savez_compressed(cost_cv_output_dir / f"cost_cv_oof_{feature}.npz",
                        alpha=COST_RIDGE_ALPHA_GRID, dollar_cost=predictions)

cost_cv_manifest = {
    "experiment": "Ridge alpha selection", "mode": RUN_MODE, "seed": SEED,
    "training_questions": cost_cv_n_questions, "model_names": model_names,
    "n_folds": len(training_folds), "alpha_grid": COST_RIDGE_ALPHA_GRID.tolist(),
    "selection_metric": "mean across 11 LLMs of per-LLM WAPE over all out-of-fold training rows",
    "normalization": "LLM cost mean from each fold's fitting rows only",
    "feature_preprocessing": "TF-IDF and auxiliary scalers refitted within each fold",
    "tie_rule": "larger alpha on exact WAPE ties",
    "selected_alpha": selected_cost_ridge_alpha,
    "calibration_used_for_selection": False, "validation_used_for_selection": False,
    "test_used": False,
    "oof_array_shape": [len(COST_RIDGE_ALPHA_GRID), cost_cv_n_questions, cost_cv_n_models],
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "elapsed_seconds": perf_counter() - cost_cv_started,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(cost_cv_output_dir / "cost_cv_manifest.json").write_text(json.dumps(cost_cv_manifest, indent=2), encoding="utf-8")

display(cost_cv_summary.style.format({
    "alpha": "{:g}", "CV_Cost_WAPE": "{:.3%}", "CV_Cost_MAE": "${:.8f}",
    "CV_Cost_MSE": "{:.3e}", "CV_Cost_MSE_normalized": "{:.6f}", "Fit_seconds": "{:.1f}",
}))
print("Selected Ridge alpha:", selected_cost_ridge_alpha)
print("Cost CV outputs:", cost_cv_output_dir)
print("Ridge alpha selection complete; the next cells fit the final score and cost predictors.")


Ridge CV / TF-IDF / fold 1: prepare features
  alpha=0.01: fold WAPE=28.503%, iterations=97, seconds=27.4
  alpha=0.1: fold WAPE=21.963%, iterations=59, seconds=18.4
  alpha=1: fold WAPE=17.786%, iterations=28, seconds=8.6
  alpha=10: fold WAPE=17.243%, iterations=13, seconds=5.2
  alpha=100: fold WAPE=21.029%, iterations=7, seconds=2.5
  alpha=1000: fold WAPE=36.969%, iterations=3, seconds=1.4
  alpha=10000: fold WAPE=56.173%, iterations=2, seconds=1.1
Ridge CV / TF-IDF / fold 2: prepare features
  alpha=0.01: fold WAPE=28.646%, iterations=84, seconds=25.2
  alpha=0.1: fold WAPE=22.075%, iterations=59, seconds=20.5
  alpha=1: fold WAPE=17.800%, iterations=26, seconds=8.9
  alpha=10: fold WAPE=17.333%, iterations=14, seconds=4.5
  alpha=100: fold WAPE=21.633%, iterations=7, seconds=2.5
  alpha=1000: fold WAPE=37.898%, iterations=3, seconds=1.3
  alpha=10000: fold WAPE=56.976%, iterations=2, seconds=1.1
Ridge CV / TF-IDF / fold 3: prepare features
  alpha=0.01: fold WAPE=28.817%, iterat

,Feature,alpha,CV_Cost_WAPE,CV_Cost_MAE,CV_Cost_MSE,CV_Cost_MSE_normalized,Fit_seconds,Selected
0,tfidf,0.01,28.655%,$0.00026508,5.119e-07,0.195960,78.9,False
1,tfidf,0.1,22.026%,$0.00021296,3.979e-07,0.140587,58.9,False
2,tfidf,1,17.710%,$0.00017437,3.565e-07,0.121314,28.9,False
3,tfidf,10,17.143%,$0.00016572,3.840e-07,0.128012,14.3,True
4,tfidf,100,21.130%,$0.00019832,4.885e-07,0.159566,7.6,False
5,tfidf,1000,37.441%,$0.00034136,8.347e-07,0.286352,4.1,False
6,tfidf,10000,56.701%,$0.00053619,1.566e-06,0.545980,3.3,False
7,e5_388,0.01,14.872%,$0.00016264,3.122e-07,0.103047,1.7,False
8,e5_388,0.1,14.786%,$0.00016158,3.120e-07,0.102982,1.4,False
9,e5_388,1,14.212%,$0.00015446,3.153e-07,0.103230,1.3,False


Selected Ridge alpha: {'tfidf': 10.0, 'e5_388': 10.0}
Cost CV outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full\cost_ridge_cv
Ridge alpha selection complete; the next cells fit the final score and cost predictors.


In [9]:
# Fit the CV-selected score classifiers; Ridge costs are fitted in the next cell.
linear_models = {}
linear_score_predictions, linear_fit_rows = {}, []
for feature in feature_labels:
    C = selected_linear_C[feature]
    X_train = split_features[feature]["train"]
    fitted_heads = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                                 max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
        joblib.delayed(fit_linear_score_job)(X_train, score_class_ids[:, position], C)
        for position in range(len(model_names))
    )
    heads = []
    for model_position, model_name in enumerate(model_names):
        head, fit_seconds = fitted_heads[model_position]
        heads.append(head)
        linear_fit_rows.append({
            "Feature": feature, "Model": model_name, "C": C,
            "Iterations": head["iterations"], "Converged": head["converged"],
            "Convergence_warnings": head["convergence_warning_count"],
            "Observed_classes": len(head["observed_classes"]), "Seconds": fit_seconds,
        })
        print(f"Logistic / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}")
    linear_models[feature] = {"heads": heads, "selected_C": C}
    for split in ("calibration", "validation"):
        X = split_features[feature][split]
        linear_score_predictions[(feature, split)] = np.stack(
            [linear_score_probabilities(head, X) for head in heads], axis=1,
        )

linear_fit_table = pd.DataFrame(linear_fit_rows)
linear_fit_table.to_csv(run_dir / "linear_final_fit.csv", index=False)
display(linear_fit_table)
if (~linear_fit_table["Converged"]).any():
    raise RuntimeError("Resolve unconverged final Logistic fits before evaluation/export; see linear_final_fit.csv.")


Logistic / TF-IDF / 1/11: WizardLM/WizardLM-13B-V1.2
Logistic / TF-IDF / 2/11: claude-instant-v1
Logistic / TF-IDF / 3/11: claude-v1
Logistic / TF-IDF / 4/11: claude-v2
Logistic / TF-IDF / 5/11: gpt-3.5-turbo-1106
Logistic / TF-IDF / 6/11: gpt-4-1106-preview
Logistic / TF-IDF / 7/11: meta/code-llama-instruct-34b-chat
Logistic / TF-IDF / 8/11: meta/llama-2-70b-chat
Logistic / TF-IDF / 9/11: mistralai/mistral-7b-chat
Logistic / TF-IDF / 10/11: mistralai/mixtral-8x7b-chat
Logistic / TF-IDF / 11/11: zero-one-ai/Yi-34B-Chat
Logistic / E5 + features (388) / 1/11: WizardLM/WizardLM-13B-V1.2
Logistic / E5 + features (388) / 2/11: claude-instant-v1
Logistic / E5 + features (388) / 3/11: claude-v1
Logistic / E5 + features (388) / 4/11: claude-v2
Logistic / E5 + features (388) / 5/11: gpt-3.5-turbo-1106
Logistic / E5 + features (388) / 6/11: gpt-4-1106-preview
Logistic / E5 + features (388) / 7/11: meta/code-llama-instruct-34b-chat
Logistic / E5 + features (388) / 8/11: meta/llama-2-70b-chat
Logi

,Feature,Model,C,Iterations,Converged,Convergence_warnings,Observed_classes,Seconds
0,tfidf,WizardLM/WizardLM-13B-V1.2,1.0,30,True,0,5,6.744792
1,tfidf,claude-instant-v1,1.0,40,True,0,5,8.929654
2,tfidf,claude-v1,1.0,31,True,0,5,6.935615
3,tfidf,claude-v2,1.0,54,True,0,5,12.507820
4,tfidf,gpt-3.5-turbo-1106,1.0,55,True,0,5,12.159888
5,tfidf,gpt-4-1106-preview,1.0,45,True,0,5,11.453214
6,tfidf,meta/code-llama-instruct-34b-chat,1.0,22,True,0,5,5.869799
7,tfidf,meta/llama-2-70b-chat,1.0,66,True,0,5,15.481355
8,tfidf,mistralai/mistral-7b-chat,1.0,41,True,0,5,8.905838
9,tfidf,mistralai/mixtral-8x7b-chat,1.0,65,True,0,5,15.429026


In [10]:
# Fit Ridge cost heads independently of Logistic score selection.
for feature in feature_labels:
    cost_model = Ridge(
        alpha=selected_cost_ridge_alpha[feature], solver="lsqr", tol=1e-3, max_iter=1000,
    ).fit(split_features[feature]["train"], train_cost_normalized)
    linear_models[feature]["cost_model"] = cost_model
    for split in ("calibration", "validation"):
        costs = np.maximum(cost_model.predict(split_features[feature][split]), 1e-6) * cost_means[None, :]
        prediction_bundles[("L2 logistic", feature, split)] = {
            "probability": linear_score_predictions[(feature, split)],
            "cost": prediction_frame(costs, split),
        }
display(pd.DataFrame({
    "Feature": list(feature_labels),
    "Applied_cost_alpha": [linear_models[feature]["cost_model"].alpha for feature in feature_labels],
}))


,Feature,Applied_cost_alpha
0,tfidf,10.0
1,e5_388,10.0


Read score-classifier CV, Ridge CV and convergence tables separately.
Logistic C and Ridge alpha are selected independently using training data.
The next methods use their own fitting cells; a complete run retrains every
predictor before the shared calibration evaluation and final export.


## 5. kNN class frequencies and mean dollar cost

Let $x$ be a question's feature vector and $x_i$ a training vector. Cosine
distance retrieves similar training questions:

$$d(x,x_i)=1-\frac{x^\mathsf{T}x_i}{\|x\|\|x_i\|}.$$

For a candidate neighbor count $k$, let $N_k(x)$ be the indices of its $k$
nearest training questions. For LLM $m$ and score class $j$, $z(s_{im})$ is
training question $i$'s mapped class and $c_{im}$ its recorded dollar cost.
The uniform neighbor estimates are

$$p_{mj}^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}\mathbf1[z(s_{im})=j],
\qquad
\hat c_m^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}c_{im}.$$

Expected score averages the mapped neighbor scores; hard class selects the
most frequent mapped class. All LLMs use the same retrieved question indices.
Queries are processed in batches, retrieving the largest requested neighborhood
once and reusing its prefixes for every candidate $k$.

Each representation receives two neighbor counts: $k_{\mathrm{score}}$ for
class probabilities and $k_{\mathrm{cost}}$ for dollar costs. Training CV selects
them independently below. The main comparison combines probabilities from
$k_{\mathrm{score}}$ with costs from $k_{\mathrm{cost}}$. Hard and expected score
outputs share these probabilities and cost predictions.


In [11]:
def _checked_knn_k_values(reference_size):
    """Validate the shared neighbor grid against the fitted reference rows."""
    if not K_VALUES or any(int(value) != value or value < 1 for value in K_VALUES):
        raise ValueError("K_VALUES must contain positive integer neighbor counts.")
    values = sorted(int(value) for value in K_VALUES)
    if len(set(values)) != len(values):
        raise ValueError("K_VALUES must not contain duplicates.")
    if values[-1] > reference_size:
        raise ValueError(
            f"The largest k ({values[-1]}) exceeds the reference size ({reference_size})."
        )
    return values


def predict_neighbor_arrays(index, features, reference_class_ids, reference_costs,
                            query_name):
    """Predict all k values from explicitly supplied, index-aligned references.

    Returns float64 probability and dollar-cost arrays. Neighbor positions refer
    to these reference arrays, not to the full training table's row positions.
    """
    reference_class_ids = np.asarray(reference_class_ids)
    reference_costs = np.asarray(reference_costs, dtype=np.float64)
    n_models, n_classes = len(model_names), len(score_support)
    if reference_class_ids.ndim != 2 or reference_class_ids.shape[1] != n_models:
        raise ValueError("Reference classes must have shape (reference rows, LLMs).")
    if not np.issubdtype(reference_class_ids.dtype, np.integer):
        raise ValueError("Reference class IDs must be integers.")
    if np.any((reference_class_ids < 0) | (reference_class_ids >= n_classes)):
        raise ValueError("Reference class IDs fall outside the score support.")
    if reference_costs.shape != reference_class_ids.shape:
        raise ValueError("Reference costs and class IDs must have identical shapes.")
    if not np.isfinite(reference_costs).all() or np.any(reference_costs <= 0):
        raise ValueError("Reference dollar costs must be finite and positive.")
    n_reference = len(reference_class_ids)
    if getattr(index, "n_samples_fit_", n_reference) != n_reference:
        raise ValueError("The fitted index and reference outcomes have different row counts.")
    k_values = _checked_knn_k_values(n_reference)
    if int(QUERY_BATCH_SIZE) != QUERY_BATCH_SIZE or QUERY_BATCH_SIZE < 1:
        raise ValueError("QUERY_BATCH_SIZE must be a positive integer.")
    batch_size = int(QUERY_BATCH_SIZE)
    count = features.shape[0]
    predictions = {
        k: {
            "probability": np.empty((count, n_models, n_classes), dtype=np.float64),
            "cost": np.empty((count, n_models), dtype=np.float64),
        }
        for k in k_values
    }
    started = perf_counter()
    max_k = k_values[-1]
    for batch_number, start in enumerate(range(0, count, batch_size), 1):
        stop = min(start + batch_size, count)
        neighbors = index.kneighbors(
            features[start:stop], n_neighbors=max_k, return_distance=False,
        )
        if neighbors.shape != (stop - start, max_k):
            raise ValueError("Unexpected neighbor-array shape.")
        labels = reference_class_ids[neighbors]
        cost_sums = np.cumsum(reference_costs[neighbors], axis=1, dtype=np.float64)
        class_counts = np.zeros((stop - start, n_models, n_classes), dtype=np.float64)
        rows, models = np.indices((stop - start, n_models))
        for position in range(max_k):
            class_counts[rows, models, labels[:, position, :]] += 1.0
            k = position + 1
            if k in predictions:
                predictions[k]["probability"][start:stop] = class_counts / k
                predictions[k]["cost"][start:stop] = cost_sums[:, position, :] / k
        if batch_number == 1 or batch_number % 10 == 0 or stop == count:
            print(
                f"kNN / {query_name}: {stop}/{count} questions | "
                f"{perf_counter() - started:.1f}s", flush=True,
            )
    return predictions


def predict_neighbor_classes(index, features, split):
    """Keep the original split-based interface for final training-reference queries."""
    predictions = predict_neighbor_arrays(
        index, features, score_class_ids,
        train_costs.loc[:, model_names].to_numpy(dtype=np.float64), split,
    )
    for bundle in predictions.values():
        bundle["cost"] = prediction_frame(bundle["cost"], split)
    return predictions


### Select neighbor counts inside training

Test $k\in\{1,2,3,5,10,20,40,60,80,100\}$ using the same three grouped
training folds as the linear models. In each fold, refit TF-IDF or the E5
auxiliary-feature scalers on the fitting rows. Build a neighbor index from
those rows only, then predict the held-out training questions. A held-out
question and every member of its question group stay outside its neighbor
library. Each training question receives one out-of-fold (OOF) prediction.

Let $N$ be the number of training questions, $m=1,\ldots,11$ index LLMs,
$z_{im}$ be the mapped recorded class, and $\hat z_{im}^{(k)}$ the OOF hard-class
prediction. Score selection maximizes macro Accuracy:

$$\mathrm{CVAccuracy}(k)=\frac1{11}\sum_{m=1}^{11}
\frac1N\sum_{i=1}^{N}\mathbf1[\hat z_{im}^{(k)}=z_{im}].$$

Let $\hat c_{im}^{(k)}$ denote the OOF dollar-cost prediction. Cost selection
minimizes macro WAPE:

$$\mathrm{CVCostWAPE}(k)=\frac1{11}\sum_{m=1}^{11}
\frac{\sum_{i=1}^{N}|\hat c_{im}^{(k)}-c_{im}|}
{\sum_{i=1}^{N}c_{im}}.$$

Pool all OOF rows before forming each LLM's WAPE. Select $k_{\mathrm{score}}$
and $k_{\mathrm{cost}}$ separately for TF-IDF and E5. Exact ties favor the
larger $k$, giving a smoother estimate. NLL and score MSE/MAE are auxiliary
diagnostics; they do not select $k$. Calibration and validation labels are
excluded from this search. A selected value at the upper boundary should be
reported before extending the grid in a later experiment.


In [12]:
# Select score and cost neighbor counts independently using training rows only.
# Prerequisites: the shared fold/preprocessing helpers and kNN helper cell.
knn_cv_k_values = _checked_knn_k_values(len(train_questions))
knn_cv_scores = train_scores.loc[:, model_names].to_numpy(dtype=np.float64)
knn_cv_costs = train_costs.loc[:, model_names].to_numpy(dtype=np.float64)
knn_cv_classes = encode_score_classes(knn_cv_scores)
knn_cv_n_questions, knn_cv_n_models = knn_cv_scores.shape
knn_cv_n_classes = len(score_support)
knn_cv_probability_floor = 1e-12
assert knn_cv_n_models == 11 and knn_cv_n_classes == 5
assert train_scores.index.equals(train_costs.index)
assert train_questions.index.equals(train_scores.index)
assert np.array_equal(knn_cv_classes, score_class_ids)
assert np.isfinite(knn_cv_costs).all() and (knn_cv_costs > 0).all()
assert len(training_folds) == 3
knn_cv_groups = train_questions["question_group"].to_numpy()
knn_cv_holdout_counts = np.zeros(knn_cv_n_questions, dtype=np.int64)
knn_cv_fold_ids = np.zeros(knn_cv_n_questions, dtype=np.int64)
for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
    assert len(fit_indices) and len(heldout_indices)
    assert np.array_equal(
        np.sort(np.concatenate([fit_indices, heldout_indices])),
        np.arange(knn_cv_n_questions),
    )
    assert not np.intersect1d(fit_indices, heldout_indices).size
    assert not np.intersect1d(
        knn_cv_groups[fit_indices], knn_cv_groups[heldout_indices],
    ).size
    assert len(fit_indices) >= max(knn_cv_k_values)
    knn_cv_holdout_counts[heldout_indices] += 1
    knn_cv_fold_ids[heldout_indices] = fold_id
assert np.all(knn_cv_holdout_counts == 1)


def knn_training_oof_metrics(probabilities, predicted_costs, recorded_scores,
                             recorded_classes, recorded_costs):
    """Return per-LLM metrics; all supplied rows belong to the training split."""
    count, n_models = recorded_scores.shape
    assert probabilities.shape == (count, n_models, len(score_support))
    assert predicted_costs.shape == recorded_costs.shape == recorded_scores.shape
    assert recorded_classes.shape == recorded_scores.shape
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1.0, atol=1e-6)
    assert np.isfinite(predicted_costs).all() and (predicted_costs > 0).all()
    hard_classes = probabilities.argmax(axis=-1)
    true_probabilities = np.take_along_axis(
        probabilities, recorded_classes[..., None], axis=-1,
    )[..., 0]
    balanced_accuracy = []
    for model_position in range(n_models):
        actual = recorded_classes[:, model_position]
        predicted = hard_classes[:, model_position]
        recalls = [
            np.mean(predicted[actual == value] == value)
            for value in np.unique(actual)
        ]
        balanced_accuracy.append(float(np.mean(recalls)))
    expected_scores = probabilities @ score_support
    cost_errors = predicted_costs - recorded_costs
    return pd.DataFrame({
        "Accuracy": (hard_classes == recorded_classes).mean(axis=0),
        "Balanced_accuracy": balanced_accuracy,
        "NLL": -np.log(np.maximum(true_probabilities, knn_cv_probability_floor)).mean(axis=0),
        "Score_MSE": np.square(expected_scores - recorded_scores).mean(axis=0),
        "Score_MAE": np.abs(expected_scores - recorded_scores).mean(axis=0),
        "Cost_MSE": np.square(cost_errors).mean(axis=0),
        "Cost_MAE": np.abs(cost_errors).mean(axis=0),
        "Cost_WAPE": np.abs(cost_errors).sum(axis=0) / recorded_costs.sum(axis=0),
    }, index=pd.Index(model_names, name="Model"))


knn_cv_output_dir = run_dir / "knn_cv"
knn_cv_output_dir.mkdir(parents=True, exist_ok=True)
knn_cv_fold_rows, knn_cv_summary_rows, knn_cv_per_llm_frames = [], [], []
selected_knn_score_k, selected_knn_cost_k = {}, {}
knn_cv_selected_rows = []
knn_cv_started = perf_counter()
knn_cv_metric_names = [
    "Accuracy", "Balanced_accuracy", "NLL", "Score_MSE", "Score_MAE",
    "Cost_MSE", "Cost_MAE", "Cost_WAPE",
]

for feature in feature_labels:
    feature_started = perf_counter()
    # Pooled tensors preserve original training-row order, independent of folds.
    oof_probabilities = np.full(
        (len(knn_cv_k_values), knn_cv_n_questions, knn_cv_n_models, knn_cv_n_classes),
        np.nan, dtype=np.float64,
    )
    oof_costs = np.full(
        (len(knn_cv_k_values), knn_cv_n_questions, knn_cv_n_models),
        np.nan, dtype=np.float64,
    )
    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        started = perf_counter()
        print(
            f"kNN training CV / {feature_labels[feature]} / fold {fold_id}: "
            f"{len(fit_indices)} fitting, {len(heldout_indices)} held-out questions",
            flush=True,
        )
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        assert X_fit.shape[0] == len(fit_indices)
        assert X_heldout.shape[0] == len(heldout_indices)
        neighbor_index = NearestNeighbors(
            metric="cosine", algorithm="brute", n_jobs=-1,
        ).fit(X_fit)
        fold_predictions = predict_neighbor_arrays(
            neighbor_index, X_heldout, knn_cv_classes[fit_indices],
            knn_cv_costs[fit_indices], f"training CV {feature} fold {fold_id}",
        )
        fold_seconds = perf_counter() - started
        for k_position, k in enumerate(knn_cv_k_values):
            bundle = fold_predictions[k]
            oof_probabilities[k_position, heldout_indices] = bundle["probability"]
            oof_costs[k_position, heldout_indices] = bundle["cost"]
            per_llm = knn_training_oof_metrics(
                bundle["probability"], bundle["cost"], knn_cv_scores[heldout_indices],
                knn_cv_classes[heldout_indices], knn_cv_costs[heldout_indices],
            )
            means = per_llm.mean(axis=0)
            knn_cv_fold_rows.append({
                "Feature": feature, "Fold": fold_id, "k": k,
                "Fit_questions": len(fit_indices),
                "Heldout_questions": len(heldout_indices),
                **{f"Fold_{name}": float(means[name]) for name in knn_cv_metric_names},
                # All k values share this retrieval/preprocessing duration.
                "Shared_fold_seconds": fold_seconds,
            })
        pd.DataFrame(knn_cv_fold_rows).to_csv(
            knn_cv_output_dir / "knn_cv_folds.csv", index=False,
        )
        del X_fit, X_heldout, neighbor_index, fold_predictions, bundle, per_llm
    assert np.isfinite(oof_probabilities).all() and np.isfinite(oof_costs).all()
    feature_rows = []
    for k_position, k in enumerate(knn_cv_k_values):
        # WAPE ratios and class recalls are formed from ALL held-out rows here,
        # rather than averaging per-fold ratios or per-fold balanced accuracies.
        per_llm = knn_training_oof_metrics(
            oof_probabilities[k_position], oof_costs[k_position],
            knn_cv_scores, knn_cv_classes, knn_cv_costs,
        )
        means = per_llm.mean(axis=0)
        feature_rows.append({
            "Feature": feature, "k": k,
            **{f"CV_{name}": float(means[name]) for name in knn_cv_metric_names},
        })
        knn_cv_per_llm_frames.append(
            per_llm.rename(columns={name: f"CV_{name}" for name in knn_cv_metric_names})
            .reset_index().assign(Feature=feature, k=k)
        )
    feature_summary = pd.DataFrame(feature_rows)
    score_choice = feature_summary.sort_values(
        ["CV_Accuracy", "k"], ascending=[False, False], kind="stable",
    ).iloc[0]
    cost_choice = feature_summary.sort_values(
        ["CV_Cost_WAPE", "k"], ascending=[True, False], kind="stable",
    ).iloc[0]
    score_k, cost_k = int(score_choice["k"]), int(cost_choice["k"])
    selected_knn_score_k[feature], selected_knn_cost_k[feature] = score_k, cost_k
    if score_k == max(knn_cv_k_values) or cost_k == max(knn_cv_k_values):
        warnings.warn(
            f"A selected k reaches the upper grid boundary for {feature}; "
            "the current result is optimal only among the tested candidates."
        )
    for row in feature_rows:
        row["Selected_score"] = row["k"] == score_k
        row["Selected_cost"] = row["k"] == cost_k
    knn_cv_summary_rows.extend(feature_rows)
    knn_cv_selected_rows.append({
        "Feature": feature, "Selected_score_k": score_k, "Selected_cost_k": cost_k,
        "Selected_CV_Accuracy": float(score_choice["CV_Accuracy"]),
        "Selected_CV_Cost_WAPE": float(cost_choice["CV_Cost_WAPE"]),
        "Shared_feature_seconds": perf_counter() - feature_started,
    })
    np.savez_compressed(
        knn_cv_output_dir / f"knn_cv_oof_{feature}.npz",
        score_probability=oof_probabilities[knn_cv_k_values.index(score_k)],
        dollar_cost=oof_costs[knn_cv_k_values.index(cost_k)],
        score_k=np.asarray(score_k, dtype=np.int64),
        cost_k=np.asarray(cost_k, dtype=np.int64),
        fold_ids=knn_cv_fold_ids,
    )
    print(
        f"kNN training CV / {feature_labels[feature]}: "
        f"score k={score_k}, cost k={cost_k}", flush=True,
    )
    del oof_probabilities, oof_costs, per_llm

knn_cv_summary = pd.DataFrame(knn_cv_summary_rows)
knn_cv_folds = pd.DataFrame(knn_cv_fold_rows)
knn_cv_per_llm = pd.concat(knn_cv_per_llm_frames, ignore_index=True)
knn_cv_per_llm = knn_cv_per_llm[[
    "Feature", "k", "Model", *[f"CV_{name}" for name in knn_cv_metric_names],
]]
knn_cv_selection = pd.DataFrame(knn_cv_selected_rows)
assert len(knn_cv_summary) == len(feature_labels) * len(knn_cv_k_values)
assert len(knn_cv_folds) == len(feature_labels) * len(training_folds) * len(knn_cv_k_values)
assert len(knn_cv_per_llm) == len(knn_cv_summary) * knn_cv_n_models
knn_cv_summary.to_csv(knn_cv_output_dir / "knn_cv_summary.csv", index=False)
knn_cv_folds.to_csv(knn_cv_output_dir / "knn_cv_folds.csv", index=False)
knn_cv_per_llm.to_csv(knn_cv_output_dir / "knn_cv_per_llm.csv", index=False)
knn_cv_selection.to_csv(knn_cv_output_dir / "knn_cv_selection.csv", index=False)
knn_cv_training_rows = train_questions.loc[:, ["sample_id", "question_group"]].copy()
knn_cv_training_rows["Fold"] = knn_cv_fold_ids
knn_cv_training_rows.to_csv(knn_cv_output_dir / "knn_cv_training_rows.csv")

knn_cv_manifest = {
    "experiment": "training-only grouped cross-validation for kNN neighbor counts",
    "mode": RUN_MODE, "seed": SEED, "training_only": True,
    "calibration_used_for_selection": False,
    "validation_used_for_selection": False, "test_used": False,
    "training_questions": knn_cv_n_questions,
    "model_names": model_names, "score_support": np.asarray(score_support).tolist(),
    "k_values": knn_cv_k_values, "n_folds": len(training_folds),
    "folds": [{
        "Fold": fold_id, "Fit_questions": len(fit_indices),
        "Heldout_questions": len(heldout_indices),
    } for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1)],
    "group_column": "question_group", "each_training_row_held_out_once": True,
    "neighbor_index": "brute-force cosine; fitting rows only",
    "feature_preprocessing": "TF-IDF and auxiliary-feature scalers refitted on each fold's fitting rows",
    "score_selection": "highest equal-LLM mean pooled out-of-fold mapped-class Accuracy",
    "cost_selection": "lowest equal-LLM mean per-LLM Cost WAPE over all out-of-fold rows",
    "metric_aggregation": "pool out-of-fold rows before per-LLM metrics; then mean the 11 LLMs",
    "numerical_score_prediction": "expected mapped score; compared with original recorded score",
    "cost_prediction": "uniform mean of original fitting-neighbor dollar costs",
    "probability_floor": knn_cv_probability_floor,
    "tie_rule": "larger k on exact primary-metric ties",
    "selected_knn_score_k": selected_knn_score_k,
    "selected_knn_cost_k": selected_knn_cost_k,
    "saved_oof_arrays": "selected score probabilities and selected cost predictions only, with their independent k values and fold IDs",
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "elapsed_seconds": perf_counter() - knn_cv_started,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(knn_cv_output_dir / "knn_cv_manifest.json").write_text(
    json.dumps(knn_cv_manifest, indent=2), encoding="utf-8",
)
display(knn_cv_summary.style.format({
    "k": "{:d}", "CV_Accuracy": "{:.2%}", "CV_Balanced_accuracy": "{:.2%}",
    "CV_NLL": "{:.5f}", "CV_Score_MSE": "{:.5f}", "CV_Score_MAE": "{:.5f}",
    "CV_Cost_MSE": "{:.3e}", "CV_Cost_MAE": "${:.8f}", "CV_Cost_WAPE": "{:.3%}",
}))
display(knn_cv_selection)
print("Selected kNN score k:", selected_knn_score_k)
print("Selected kNN cost k:", selected_knn_cost_k)
print("Training-only kNN CV outputs:", knn_cv_output_dir)


kNN training CV / TF-IDF / fold 1: 14601 fitting, 7299 held-out questions
kNN / training CV tfidf fold 1: 128/7299 questions | 0.6s
kNN / training CV tfidf fold 1: 1280/7299 questions | 8.4s
kNN / training CV tfidf fold 1: 2560/7299 questions | 17.6s
kNN / training CV tfidf fold 1: 3840/7299 questions | 26.8s
kNN / training CV tfidf fold 1: 5120/7299 questions | 35.1s
kNN / training CV tfidf fold 1: 6400/7299 questions | 43.2s
kNN / training CV tfidf fold 1: 7299/7299 questions | 49.1s
kNN training CV / TF-IDF / fold 2: 14602 fitting, 7298 held-out questions
kNN / training CV tfidf fold 2: 128/7298 questions | 0.6s
kNN / training CV tfidf fold 2: 1280/7298 questions | 8.2s
kNN / training CV tfidf fold 2: 2560/7298 questions | 17.4s
kNN / training CV tfidf fold 2: 3840/7298 questions | 28.3s
kNN / training CV tfidf fold 2: 5120/7298 questions | 36.3s
kNN / training CV tfidf fold 2: 6400/7298 questions | 44.4s
kNN / training CV tfidf fold 2: 7298/7298 questions | 50.5s
kNN training CV / 

,Feature,k,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Score_MSE,CV_Score_MAE,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,Selected_score,Selected_cost
0,tfidf,1,60.39%,43.39%,10.94413,0.29944,0.32470,7.012e-07,$0.00023105,25.632%,False,False
1,tfidf,2,58.41%,43.19%,6.27374,0.22727,0.32502,6.165e-07,$0.00021897,24.450%,False,False
2,tfidf,3,63.24%,43.54%,4.22483,0.20246,0.32503,5.879e-07,$0.00021469,24.032%,False,False
3,tfidf,5,64.62%,43.69%,2.54546,0.18272,0.32588,5.606e-07,$0.00021274,23.952%,False,True
4,tfidf,10,66.05%,43.92%,1.37034,0.16794,0.32670,5.457e-07,$0.00021454,24.252%,False,False
5,tfidf,20,67.00%,43.75%,0.91652,0.16129,0.32844,5.482e-07,$0.00022185,25.217%,False,False
6,tfidf,40,67.15%,43.44%,0.75998,0.15862,0.33034,5.531e-07,$0.00023246,26.711%,False,False
7,tfidf,60,67.21%,43.34%,0.72074,0.15796,0.33149,5.541e-07,$0.00023801,27.559%,False,False
8,tfidf,80,67.22%,43.20%,0.70611,0.15778,0.33230,5.559e-07,$0.00024191,28.138%,True,False
9,tfidf,100,67.20%,43.07%,0.69988,0.15777,0.33317,5.575e-07,$0.00024504,28.586%,False,False


,Feature,Selected_score_k,Selected_cost_k,Selected_CV_Accuracy,Selected_CV_Cost_WAPE,Shared_feature_seconds
0,tfidf,80,5,0.672154,0.239518,205.591949
1,e5_388,80,10,0.679199,0.103515,16.754796


Selected kNN score k: {'tfidf': 80, 'e5_388': 80}
Selected kNN cost k: {'tfidf': 5, 'e5_388': 10}
Training-only kNN CV outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full\knn_cv


In [13]:
knn_indices, knn_class_predictions = {}, {}
for feature in feature_labels:
    score_k = selected_knn_score_k[feature]
    cost_k = selected_knn_cost_k[feature]
    index = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(
        split_features[feature]["train"],
    )
    knn_indices[feature] = index
    knn_class_predictions[feature] = {}
    for split in ("calibration", "validation"):
        all_k = predict_neighbor_classes(index, split_features[feature][split], split)
        knn_class_predictions[feature][split] = all_k
        prediction_bundles[("kNN", feature, split)] = {
            "probability": all_k[score_k]["probability"],
            "cost": all_k[cost_k]["cost"],
        }
    print(f"Selected kNN / {feature_labels[feature]}: score k={score_k}, cost k={cost_k}")


kNN / calibration: 128/3648 questions | 1.1s
kNN / calibration: 1280/3648 questions | 13.2s
kNN / calibration: 2560/3648 questions | 26.1s
kNN / calibration: 3648/3648 questions | 36.2s
kNN / validation: 128/3649 questions | 1.0s
kNN / validation: 1280/3649 questions | 13.4s
kNN / validation: 2560/3649 questions | 26.9s
kNN / validation: 3649/3649 questions | 37.1s
Selected kNN / TF-IDF: score k=80, cost k=5
kNN / calibration: 128/3648 questions | 0.3s
kNN / calibration: 1280/3648 questions | 1.1s
kNN / calibration: 2560/3648 questions | 2.2s
kNN / calibration: 3648/3648 questions | 3.1s
kNN / validation: 128/3649 questions | 0.1s
kNN / validation: 1280/3649 questions | 1.1s
kNN / validation: 2560/3649 questions | 2.2s
kNN / validation: 3649/3649 questions | 3.2s
Selected kNN / E5 + features (388): score k=80, cost k=10


## 6. XGBoost score classification and price prediction

Fit one boosted-tree score classifier and one price model per LLM. The score
head uses multiclass cross-entropy (`multi:softprob`). If a training subset has
only two observed classes, it uses binary cross-entropy. A constant target
produces a constant distribution. In every case, align the output to the same
five class columns and fill unobserved classes with zero probability.

The price head minimizes squared error on normalized costs. Restore dollar
units with the training cost means. Both hard and expected score outputs share
these price predictions. Tree settings are fixed in this experiment; XGBoost
hyperparameters are not selected by cross-validation here.


In [14]:
xgb_params = {
    "n_estimators": XGB_ROUNDS, "max_depth": 3, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 64,
    "tree_method": "hist", "device": XGB_DEVICE, "n_jobs": XGB_N_JOBS,
    "reg_lambda": 1.0, "random_state": SEED,
}


def fit_xgb_score_head(features, targets):
    classes, encoded = np.unique(targets, return_inverse=True)
    if len(classes) == 1:
        return {"classifier": None, "classes": classes}
    objective = "binary:logistic" if len(classes) == 2 else "multi:softprob"
    options = {"eval_metric": "logloss" if len(classes) == 2 else "mlogloss"}
    if len(classes) > 2:
        options["num_class"] = len(classes)
    classifier = XGBClassifier(objective=objective, **options, **xgb_params).fit(features, encoded)
    actual_device = json.loads(classifier.get_booster().save_config())["learner"]["generic_param"]["device"]
    if XGB_DEVICE == "cuda" and not actual_device.startswith("cuda"):
        raise RuntimeError("XGBoost fell back to CPU. Set XGB_DEVICE='cpu' or check the GPU installation.")
    return {"classifier": classifier, "classes": classes}


def xgb_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["classes"][0]] = 1.0
        return probabilities
    return align_class_probabilities(head["classifier"].predict_proba(features), head["classes"], len(score_support))


def fit_xgb_pair(features, score_targets, cost_targets):
    """Fit the independent score/cost pair for one candidate LLM."""
    started = perf_counter()
    head = fit_xgb_score_head(features, score_targets)
    price_head = XGBRegressor(objective="reg:squarederror", **xgb_params).fit(
        features, cost_targets,
    )
    return head, price_head, perf_counter() - started


In [15]:
xgb_models = {}
for feature in feature_labels:
    X_train = split_features[feature]["train"]
    xgb_models[feature] = {"score_classifier": [], "cost_regressor": []}
    raw = {split: {"probability": [], "cost": []} for split in ("calibration", "validation")}
    print(f"XGBoost / {feature_labels[feature]}: fitting {len(model_names)} score/cost pairs", flush=True)
    fitted_pairs = joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes",
                           max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
        joblib.delayed(fit_xgb_pair)(
            X_train, score_class_ids[:, position], train_cost_normalized[:, position],
        ) for position in range(len(model_names))
    )
    for model_position, model_name in enumerate(model_names):
        print(f"XGBoost / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}", flush=True)
        head, price_head, fit_seconds = fitted_pairs[model_position]
        xgb_models[feature]["score_classifier"].append(head)
        xgb_models[feature]["cost_regressor"].append(price_head)
        for split in raw:
            X = split_features[feature][split]
            raw[split]["probability"].append(xgb_score_probabilities(head, X))
            costs = np.maximum(price_head.predict(X), 1e-6) * cost_means[model_position]
            raw[split]["cost"].append(costs)
        print(f"  completed in {fit_seconds:.1f}s", flush=True)
    for split, values in raw.items():
        prediction_bundles[("XGBoost", feature, split)] = {
            "probability": np.stack(values["probability"], axis=1),
            "cost": prediction_frame(np.column_stack(values["cost"]), split),
        }


XGBoost / TF-IDF: fitting 11 score/cost pairs
XGBoost / TF-IDF / 1/11: WizardLM/WizardLM-13B-V1.2
  completed in 307.2s
XGBoost / TF-IDF / 2/11: claude-instant-v1
  completed in 301.8s
XGBoost / TF-IDF / 3/11: claude-v1
  completed in 306.3s
XGBoost / TF-IDF / 4/11: claude-v2
  completed in 301.9s
XGBoost / TF-IDF / 5/11: gpt-3.5-turbo-1106
  completed in 307.4s
XGBoost / TF-IDF / 6/11: gpt-4-1106-preview
  completed in 296.7s
XGBoost / TF-IDF / 7/11: meta/code-llama-instruct-34b-chat
  completed in 303.9s
XGBoost / TF-IDF / 8/11: meta/llama-2-70b-chat
  completed in 307.2s
XGBoost / TF-IDF / 9/11: mistralai/mistral-7b-chat
  completed in 293.0s
XGBoost / TF-IDF / 10/11: mistralai/mixtral-8x7b-chat
  completed in 311.3s
XGBoost / TF-IDF / 11/11: zero-one-ai/Yi-34B-Chat
  completed in 227.1s
XGBoost / E5 + features (388): fitting 11 score/cost pairs
XGBoost / E5 + features (388) / 1/11: WizardLM/WizardLM-13B-V1.2
  completed in 17.2s
XGBoost / E5 + features (388) / 2/11: claude-instant-

## 7. Evaluate probabilities and the two score outputs

**Evaluation split:** all predictor comparisons here use calibration.
Training-only grouped CV selects Logistic C, Ridge alpha, and separate kNN
score/cost neighbor counts. Use calibration to compare the selected predictors,
input representations and hard/expected score outputs. Validation predictions are exported for downstream routing-policy
development; this section does not evaluate validation outcomes.

The displayed tables use the freshly fitted models and their calibration
predictions. Preserve the untouched test
split for evaluating the fixed complete system.

For question $i$ and candidate LLM $m$:

$$\hat s^{\mathrm{hard}}_{im}=a_{\arg\max_j p_{imj}},
\qquad \hat s^{\mathrm{expected}}_{im}=\sum_{j=0}^4 a_jp_{imj}.$$

Hard class belongs to the five anchors. Expected score can take an intermediate
value, such as 0.63. An exact probability tie uses the lower anchor because the
class columns are ordered from 0 to 1.

**Probability metrics** evaluate the five-class target $z_{im}$:

$$\mathrm{Accuracy}_m=\frac1n\sum_i
\mathbf1[\arg\max_j p_{imj}=z_{im}],
\qquad
\mathrm{NLL}_m=-\frac1n\sum_i\log\max(p_{im,z_{im}},10^{-12}).$$

Balanced accuracy averages recall across the classes present in calibration.
It reveals performance hidden by frequent classes. Report the fraction of
zero true-class probabilities alongside NLL; the numerical floor keeps NLL
finite. Accuracy measures hard class decisions, while NLL evaluates the whole
distribution.

**Numerical error metrics** compare both score outputs with the original recorded
score $s_{im}$. Predicted costs are compared with recorded dollar costs $c_{im}$.
For MSE and MAE below, $y_{im}$ denotes either the original score or the
recorded cost, and $\hat y_{im}$ is the corresponding prediction:

$$\mathrm{MSE}_m=\frac1n\sum_i(\hat y_{im}-y_{im})^2,
\qquad \mathrm{MAE}_m=\frac1n\sum_i|\hat y_{im}-y_{im}|,$$

**Cost WAPE** is used only for dollar-cost prediction:

$$\mathrm{CostWAPE}_m=
\frac{\sum_i|\hat c_{im}-c_{im}|}{\sum_i c_{im}}.$$

Compute each metric separately for each LLM, then average equally across the
11 LLMs. Score prediction is evaluated with MSE and MAE on its original 0-to-1
scale. Cost prediction uses MSE, MAE and WAPE. All recorded costs are positive.
Hard and expected outputs share price predictions, so their cost errors are
identical. Probability metrics above remain unchanged.


In [16]:
NLL_EPSILON = 1e-12
score_support = np.asarray(score_support, dtype=float)
assert np.all(np.diff(score_support) > 0), "The score support must be strictly sorted."
evaluation_index = evaluation_scores.index
assert evaluation_costs.index.equals(evaluation_index), "Calibration score/cost rows must align."
evaluation_truth = evaluation_scores.loc[:, model_names].to_numpy(dtype=float)
evaluation_cost_truth = evaluation_costs.loc[:, model_names].to_numpy(dtype=float)
n_questions, n_models = evaluation_truth.shape
n_score_classes = len(score_support)


evaluation_class_ids = encode_score_classes(evaluation_truth)


def checked_probability_bundle(bundle):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    costs = bundle["cost"]
    if probabilities.shape != (n_questions, n_models, n_score_classes):
        raise ValueError("Probabilities must have shape (questions, candidate LLMs, classes).")
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0):
        raise ValueError("Class probabilities must be finite and nonnegative.")
    if not np.allclose(probabilities.sum(axis=-1), 1.0, atol=1e-6):
        raise ValueError("Each candidate LLM needs a probability distribution summing to one.")
    if not costs.index.equals(evaluation_index) or costs.columns.tolist() != model_names:
        raise ValueError("Cost prediction rows and model columns must preserve calibration order.")
    cost_values = costs.to_numpy(dtype=float)
    if not np.isfinite(cost_values).all() or np.any(cost_values < 0):
        raise ValueError("Predicted dollar costs must be finite and nonnegative.")
    return probabilities, cost_values


def distribution_metrics(bundle):
    probabilities, _ = checked_probability_bundle(bundle)
    hard_class_ids = probabilities.argmax(axis=-1)
    true_probabilities = np.take_along_axis(
        probabilities, evaluation_class_ids[..., None], axis=-1,
    )[..., 0]
    balanced_accuracy = []
    for model_position in range(n_models):
        truth = evaluation_class_ids[:, model_position]
        prediction = hard_class_ids[:, model_position]
        recalls = [(prediction[truth == value] == value).mean() for value in np.unique(truth)]
        balanced_accuracy.append(float(np.mean(recalls)))
    return pd.DataFrame({
        "Accuracy": (hard_class_ids == evaluation_class_ids).mean(axis=0),
        "Balanced_accuracy": balanced_accuracy,
        "NLL": -np.log(np.maximum(true_probabilities, NLL_EPSILON)).mean(axis=0),
        "Zero_true_probability_rate": (true_probabilities == 0).mean(axis=0),
    }, index=pd.Index(model_names, name="model"))


def score_outputs(bundle):
    probabilities, costs = checked_probability_bundle(bundle)
    expected_scores = probabilities @ score_support
    return {
        "Expected score": {"score": expected_scores, "cost": costs},
        "Hard class": {"score": score_support[probabilities.argmax(axis=-1)], "cost": costs},
    }


def cost_wape(predicted_costs, recorded_costs):
    """Return one aggregate relative dollar-cost error for each LLM."""
    totals = np.abs(recorded_costs).sum(axis=0)
    return np.divide(
        np.abs(predicted_costs - recorded_costs).sum(axis=0), totals,
        out=np.full(recorded_costs.shape[1], np.nan), where=totals > 0,
    )


def numeric_prediction_metrics(bundle):
    scores, costs = bundle["score"], bundle["cost"]
    return pd.DataFrame({
        "Score_MSE": np.square(scores - evaluation_truth).mean(axis=0),
        "Score_MAE": np.abs(scores - evaluation_truth).mean(axis=0),
        "Cost_MSE": np.square(costs - evaluation_cost_truth).mean(axis=0),
        "Cost_MAE": np.abs(costs - evaluation_cost_truth).mean(axis=0),
        "Cost_WAPE": cost_wape(costs, evaluation_cost_truth),
    }, index=pd.Index(model_names, name="model"))


def training_frequency_bundle(train_scores, train_costs):
    train_values = train_scores.loc[:, model_names].to_numpy(dtype=float)
    train_classes = encode_score_classes(train_values)
    frequencies = np.stack([
        np.bincount(train_classes[:, model], minlength=n_score_classes) / len(train_values)
        for model in range(n_models)
    ])
    mean_costs = train_costs.loc[:, model_names].mean(axis=0).to_numpy(dtype=float)
    return {
        "probability": np.broadcast_to(frequencies, (n_questions, n_models, n_score_classes)),
        "cost": pd.DataFrame(np.broadcast_to(mean_costs, (n_questions, n_models)),
                             index=evaluation_index, columns=model_names),
    }


In [17]:
# prediction_bundles is keyed by (method, embedding, split).
evaluation_probability_bundles = {
    (method, embedding): bundle
    for (method, embedding, split), bundle in prediction_bundles.items()
    if split == EVALUATION_SPLIT
}
evaluation_probability_bundles = {
    ("Training frequency", "Baseline"): training_frequency_bundle(train_scores, train_costs),
    **evaluation_probability_bundles,
}
distribution_per_model = pd.concat({
    key: distribution_metrics(bundle)
    for key, bundle in evaluation_probability_bundles.items()
}, names=["method", "embedding"])
distribution_summary = distribution_per_model.groupby(
    level=["method", "embedding"], sort=False,
).mean()
numeric_evaluation_bundles = {
    (method, embedding, output): values
    for (method, embedding), bundle in evaluation_probability_bundles.items()
    for output, values in score_outputs(bundle).items()
}
numeric_per_model = pd.concat({
    key: numeric_prediction_metrics(bundle)
    for key, bundle in numeric_evaluation_bundles.items()
}, names=["method", "embedding", "score_output"])
numeric_summary = numeric_per_model.groupby(
    level=["method", "embedding", "score_output"], sort=False,
).mean()

print("Calibration probability quality: one result per trained classifier")
display(distribution_summary.style.format({
    "Accuracy": "{:.1%}", "Balanced_accuracy": "{:.1%}", "NLL": "{:.4f}",
    "Zero_true_probability_rate": "{:.1%}",
}))
print("Calibration score/cost errors: two score outputs from the same probabilities")
display(numeric_summary.style.format({
    "Score_MSE": "{:.5f}", "Score_MAE": "{:.5f}",
    "Cost_MSE": "{:.3e}", "Cost_MAE": "${:.8f}", "Cost_WAPE": "{:.1%}",
}, na_rep="undefined"))
distribution_summary.to_csv(run_dir / f"{EVALUATION_SPLIT}_classification_summary.csv")
distribution_per_model.to_csv(run_dir / f"{EVALUATION_SPLIT}_classification_per_llm.csv")
numeric_summary.to_csv(run_dir / f"{EVALUATION_SPLIT}_numeric_summary.csv")
numeric_per_model.to_csv(run_dir / f"{EVALUATION_SPLIT}_numeric_per_llm.csv")


Calibration probability quality: one result per trained classifier


Calibration score/cost errors: two score outputs from the same probabilities


## 8. Save and reuse the results

Each run saves probability arrays, hard scores, expected scores and dollar costs
for calibration and validation. DataFrames preserve question indices and LLM
columns. Probability arrays use `(questions, 11 LLMs, 5 classes)`; the manifest
records both column orders.

`predictions_manifest.csv` identifies the main six predictor/feature combinations.
Its kNN rows record `score_k` and `cost_k` separately. Use these prefixes for the
CV-selected main predictor. `knn_predictions_manifest.csv` additionally exports
every candidate k for both splits; each of those grid bundles uses the same k
for scores and cost. Its selection flags show which component was chosen.

The `knn_cv/` folder contains train-only fold, pooled OOF, per-LLM and selection
records. Calibration metric CSVs evaluate the selected predictors; validation
prediction files remain available for downstream routing-policy development.

The folder also contains class-coverage tables, linear-model CV records, model
settings and fitted Logistic/Ridge/XGBoost objects. Prediction on new questions
requires the same prepared TF-IDF/E5 workflow. Rebuild kNN indices from the
prepared training features, mapped score labels and recorded cost labels, using
the selected score and cost k values in the manifest.

For full runs, all files below belong to the fixed `full/` directory. Each
complete rerun replaces its old contents. Smoke outputs remain separate.
`run_status.json` records the actual run state and exported prediction count;
`experiment_manifest.json` records the selected parameters for that run.
The training-frequency baseline appears in calibration metric CSVs; it has no
separate prediction bundle. The prepared Step 1 files are kept outside this
output directory and are required again for new-text inference.


In [18]:
def save_prediction_bundle(prefix, bundle, split):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    assert probabilities.shape == (len(tables[split]["questions"]), len(model_names), 5)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1, atol=1e-6)
    np.save(run_dir / f"{prefix}_probability.npy", probabilities, allow_pickle=False)
    prediction_frame(probabilities @ score_support, split).to_pickle(run_dir / f"{prefix}_expected_score.pkl")
    prediction_frame(score_support[probabilities.argmax(axis=-1)], split).to_pickle(run_dir / f"{prefix}_hard_score.pkl")
    costs = bundle["cost"]
    assert costs.index.equals(tables[split]["questions"].index)
    assert costs.columns.tolist() == model_names
    assert np.isfinite(costs.to_numpy(dtype=float)).all() and costs.gt(0).all().all()
    costs.to_pickle(run_dir / f"{prefix}_dollar_cost.pkl")


prediction_manifest = []
for position, ((method, feature, split), bundle) in enumerate(prediction_bundles.items()):
    prefix = f"pred_{position:02d}_{split}"
    save_prediction_bundle(prefix, bundle, split)
    prediction_manifest.append({
        "prefix": prefix, "method": method, "embedding": feature, "split": split,
        "k": (selected_knn_score_k[feature]
              if method == "kNN" and selected_knn_score_k[feature] == selected_knn_cost_k[feature]
              else None),
        "score_k": selected_knn_score_k[feature] if method == "kNN" else None,
        "cost_k": selected_knn_cost_k[feature] if method == "kNN" else None,
    })
pd.DataFrame(prediction_manifest).to_csv(run_dir / "predictions_manifest.csv", index=False)

knn_prediction_manifest = []
for feature, split_predictions in knn_class_predictions.items():
    for split, all_k in split_predictions.items():
        for k, bundle in all_k.items():
            prefix = f"knn_{feature}_k{k}_{split}"
            save_prediction_bundle(prefix, bundle, split)
            knn_prediction_manifest.append({
                "prefix": prefix, "embedding": feature, "split": split, "k": k,
                "selected_for_score": k == selected_knn_score_k[feature],
                "selected_for_cost": k == selected_knn_cost_k[feature],
            })
pd.DataFrame(knn_prediction_manifest).to_csv(run_dir / "knn_predictions_manifest.csv", index=False)

linear_cv_table.to_csv(run_dir / "linear_cv_folds.csv", index=False)
linear_cv_head_table.to_csv(run_dir / "linear_cv_heads.csv", index=False)
linear_cv_summary.to_csv(run_dir / "linear_cv_summary.csv", index=False)
linear_fit_table.to_csv(run_dir / "linear_final_fit.csv", index=False)
joblib.dump(linear_models, run_dir / "linear_models.joblib")
joblib.dump(xgb_models, run_dir / "xgb_models.joblib")

manifest = {
    "mode": RUN_MODE, "seed": SEED,
    "execution_provenance": dict(execution_provenance),
    "model_names": model_names, "score_support": score_support.tolist(),
    "label_mapping": "nearest fixed anchor; exact midpoint uses lower anchor",
    "numerical_score_metrics_target": "original recorded score",
    "training_cost_means": cost_means.tolist(),
    "selected_linear_C": selected_linear_C,
    "linear_C_grid": LINEAR_C_GRID.tolist(),
    "linear_C_selection": {"data": "train", "folds": len(training_folds),
                           "metric": "mean out-of-fold NLL across questions and LLMs",
                           "exact_tie_rule": "smaller C"},
    # Record the applied penalties from the newly fitted cost models.
    "cost_ridge_alpha": {feature: float(models["cost_model"].alpha)
                         for feature, models in linear_models.items()},
    "selected_cost_ridge_alpha": selected_cost_ridge_alpha,
    "cost_ridge_cv_metric": "mean per-LLM out-of-fold training Cost WAPE",
    "fit_n_jobs": FIT_N_JOBS, "xgb_parameters": xgb_params, "k_values": K_VALUES,
    "selected_knn_score_k": selected_knn_score_k,
    "selected_knn_cost_k": selected_knn_cost_k,
    "knn_selection": {"data": "train", "folds": len(training_folds),
                      "score_metric": "macro OOF Accuracy",
                      "cost_metric": "macro pooled OOF Cost WAPE",
                      "exact_tie_rule": "larger k"},
    "rows": {split: len(tables[split]["questions"]) for split in SPLITS},
    "test_used": False, "calibration_fitted": False,
    "evaluation_split": EVALUATION_SPLIT,
    "evaluation_purpose": "base-predictor development and diagnostics",
    "score_error_metrics": ["Score_MSE", "Score_MAE"],
    "cost_error_metrics": ["Cost_MSE", "Cost_MAE", "Cost_WAPE"],
    "validation_role": "downstream routing-policy development",
    "versions": {"sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(run_dir / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
# Mark completion only after every expected prediction file has been saved.
assert len(prediction_manifest) == 3 * len(feature_labels) * 2
assert len(knn_prediction_manifest) == len(feature_labels) * 2 * len(K_VALUES)
prediction_prefixes = [row["prefix"] for row in prediction_manifest + knn_prediction_manifest]
assert len(prediction_prefixes) == len(set(prediction_prefixes))
prediction_suffixes = ("_probability.npy", "_expected_score.pkl", "_hard_score.pkl", "_dollar_cost.pkl")
prediction_paths = [run_dir / f"{prefix}{suffix}"
                    for prefix in prediction_prefixes for suffix in prediction_suffixes]
assert all(path.is_file() for path in prediction_paths)
run_status = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
run_status.update({
    "status": "completed", "completed_utc": datetime.now(timezone.utc).isoformat(),
    "refit_scope": "all_predictors", "prediction_files": len(prediction_paths),
    "rows": {split: len(tables[split]["questions"]) for split in SPLITS},
})
(run_dir / "run_status.json").write_text(json.dumps(run_status, indent=2), encoding="utf-8")

print("Completed mode:", RUN_MODE)
print("Fixed score classes:", score_support.tolist())
print("Experiment outputs:", run_dir)
if RUN_MODE == "smoke":
    print("Smoke validates execution. Use full mode for performance comparisons.")


Completed mode: full
Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]
Experiment outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full


## 9. Read the results

- Compare accuracy, balanced accuracy and NLL on the same calibration rows.
- Compare hard and expected outputs with raw-score MSE and MAE.
- Inspect per-LLM CSVs when an average hides a weak candidate model.
- Use the training-frequency baseline to assess what the question features add.
- Select C, alpha and k within training, then compare the selected predictors
  on calibration before developing the routing policy on validation.
- Evaluate the selected complete system on the untouched test split.

These are development diagnostics. Earlier exploration repeatedly inspected
validation results; moving the current comparisons to calibration does not
undo that history or create a new independent final estimate.
